# Tutorial: Multislice Propagation and Aperture ROI Modes

This notebook explains the difference between running the hologram simulation with and without multislice free-space propagation, and how aperture regions of interest (ROIs) can be used in different parts of the pipeline.

The key idea:

- **Jones interaction** applies each material slice locally to the beam field.
- **Multislice propagation** additionally propagates the field through free space between material slices.
- **Aperture ROIs** restrict expensive operations to the object/reference-hole support where the mask exposes the sample.

The notebook is didactic first. It includes a compact mode table and an optional tiny pipeline comparison at the end.


## 1. Imports

In [ ]:
import sys
from pathlib import Path
from dataclasses import fields, replace
import time

import h5py
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    %matplotlib widget
except Exception:
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from fomocid import DATA_ROOT
from scattering_calculator.simulation_pipelines import (
    FrontApertureConfig,
    SampleConfig,
    XRayConfig,
)
from scattering_calculator.simulation_pipelines.pipelines import (
    HologramPipeline,
    HologramPipelineConfig,
    HologramPipelineRanges,
)
from scattering_calculator.utils import physics


## 2. What changes when `propagate` changes?

In `HologramPipelineConfig`, the flag is:

```python
propagate=False  # local Jones interaction only
propagate=True   # local Jones interaction plus free-space propagation between slices
```

When `propagate=False`, each layer modifies the local Jones vector according to the material dielectric tensor and layer thickness. No angular-spectrum propagation is applied between layers. This is faster and often adequate when the stack is thin and near-field spreading within the stack is negligible.

When `propagate=True`, the field is propagated through free space after each material slice except the last one. This is closer to a multislice treatment: layer transmission, propagation, layer transmission, propagation, and so on. It is more expensive because each free-space step uses FFT-based propagation.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.set_axis_off()
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)

layer_x = [3.0, 6.0, 9.0]
row_y = {"propagate=False": 3.35, "propagate=True": 1.35}
box_w, box_h = 1.15, 0.75

# Row labels live in a separate left margin, away from arrows and layer boxes.
for label, y in row_y.items():
    ax.text(0.45, y + box_h / 2, label, va="center", ha="left", fontweight="bold")

# Draw the material layers.
for row_label, y in row_y.items():
    for i, x in enumerate(layer_x):
        ax.add_patch(
            plt.Rectangle(
                (x, y),
                box_w,
                box_h,
                facecolor="tab:blue",
                alpha=0.32,
                edgecolor="black",
                linewidth=1.2,
            )
        )
        ax.text(x + box_w / 2, y + box_h / 2, f"layer {i}", ha="center", va="center")

# Draw arrows between layers with labels offset from the arrow shaft.
def arrow_between(x0, x1, y, label, label_dy):
    start = (x0 + box_w + 0.15, y + box_h / 2)
    end = (x1 - 0.15, y + box_h / 2)
    ax.annotate("", xy=end, xytext=start, arrowprops={"arrowstyle": "->", "lw": 1.4})
    ax.text((start[0] + end[0]) / 2, start[1] + label_dy, label, ha="center", va="center", fontsize=9)

# Jones-only mode: local interaction at each layer, no inter-layer free-space FFT.
arrow_between(layer_x[0], layer_x[1], row_y["propagate=False"], "next layer", 0.35)
arrow_between(layer_x[1], layer_x[2], row_y["propagate=False"], "next layer", 0.35)
ax.text(layer_x[2] + box_w + 0.35, row_y["propagate=False"] + box_h / 2, "exit wave", va="center")

# Multislice mode: local Jones interaction plus free-space propagation between layers.
arrow_between(layer_x[0], layer_x[1], row_y["propagate=True"], "free-space FFT", -0.38)
arrow_between(layer_x[1], layer_x[2], row_y["propagate=True"], "free-space FFT", -0.38)
ax.text(layer_x[2] + box_w + 0.35, row_y["propagate=True"] + box_h / 2, "exit wave", va="center")

ax.text(6.0, 4.65, "Local Jones-only propagation versus multislice free-space propagation", ha="center", fontsize=12, fontweight="bold")
ax.text(6.0, 4.28, "Each blue box is a local Jones/material interaction; only the lower row propagates through free space between boxes.", ha="center", fontsize=9)


## 3. ROI flags in the pipeline

The pipeline has one master ROI switch and several more specific switches.

| Config flag | Affects | Meaning |
|---|---|---|
| `use_roi` | master switch | If `False`, ROI accelerations are disabled even if the specific flags are `True`. |
| `magnetic_pattern_use_roi` | pattern generation | Generate expensive magnetic patterns only around the object-hole region, then paste into a full field. |
| `dielectric_tensor_use_roi` | dielectric tensor and Jones interaction | Build aperture support regions and compute dense/magnetic/vacuum corrections only where the FTH mask opens holes. |
| `dielectric_tensor_compact` | memory representation | Store constant per-layer diagonal terms plus ROI patches instead of one dense `(Nz, Ny, Nx, 2, 2)` array. |
| `multislice_propagation_roi` | free-space propagation between slices | If `propagate=True`, run FFT propagation only inside aperture ROI boxes and add each crop's correction to the plane-wave baseline field. |
| `multislice_propagation_roi_padding_px` | multislice ROI boxes | Enlarge each aperture ROI before local free-space propagation. |
| `multislice_propagation_roi_merge_overlaps` | padded multislice ROI boxes | If `True`, merge overlapping padded ROI boxes before local propagation. Safer for close OH/RH layouts, but potentially slower. |

Important nuance: **these ROIs are computational regions, not hard masks**. They do not zero the exit wave outside the boxes. A nonzero exit-wave background outside the ROI rectangles is expected and does not by itself mean ROI mode was ignored.

### What happens outside each ROI?

The word ROI is used in three different places, and each one has a different outside-ROI approximation.

| ROI use | Where the full calculation is done | What happens outside the ROI | Approximation implied |
|---|---|---|---|
| `magnetic_pattern_use_roi=True` | The expensive magnetic texture generator runs on one padded object-hole bounding box for supported pattern types such as labyrinth, wavy stripe, and disordered skyrmion patterns. | The full magnetic-pattern array is initialized to `+1`, then the generated local texture is pasted into the object-hole ROI. Outside the ROI the magnetization therefore stays at the background value. | This assumes magnetic texture outside the object-hole bounding box is irrelevant for the simulated hologram, because it is hidden by the aperture mask or not saved in the object-hole-focused diagnostic output. |
| `dielectric_tensor_use_roi=True` | Aperture support regions are built around the FTH holes. Dense magnetic/vacuum dielectric corrections are computed inside those support boxes. | Outside the support boxes, the material slice is treated as spatially uniform and diagonal for each layer. During Jones propagation the outside pixels receive the constant per-layer diagonal transmission, using the layer background terms. They are not set to zero. | This assumes the expensive off-diagonal/magnetic/aperture corrections only matter near the apertures, while the masked/background material outside can be represented by the constant layer response. |
| `multislice_propagation_roi=True` with `propagate=True` | The free-space angular-spectrum FFT between material slices is evaluated only inside padded aperture support boxes. | The whole field starts from the zero-spatial-frequency plane-wave phase `exp(-i k0 dz)`. Each ROI crop adds its local correction, `local_propagated - plane_wave_baseline`, back into the full field. With `multislice_propagation_roi_merge_overlaps=True`, overlapping padded boxes are merged first so nearby apertures are propagated as one local crop. Outside all boxes there is no diffraction/spreading calculation, only the plane-wave phase advance. | This is the strongest approximation. True free-space propagation is nonlocal, so light can diffract between ROI and non-ROI pixels. Increase `multislice_propagation_roi_padding_px`, enable ROI merging, or use full-field multislice when that coupling matters. |

Consequences for the plots below:

- ROI rectangles indicate where the expensive work was focused; they are not expected to bound all nonzero signal.
- In Jones-only ROI mode, outside-ROI pixels still carry the incoming illumination multiplied by the background layer transmission.
- In ROI-multislice mode, outside-ROI pixels still carry the incoming field phase-advanced through the inter-slice distance. What is missing outside the ROI is local diffractive redistribution.
- If you need the outside field to include full diffraction from every pixel to every other pixel, use `propagate=True` and `multislice_propagation_roi=False`.

**ROI-only multislice free-space propagation is an approximation**. Free-space propagation is non-local, so a full-field FFT is the more physically faithful option. ROI-only multislice is faster when most of the field is covered by the mask and only aperture regions matter.


## 4. The main operating modes

Use this table as a practical map:

| Mode | `propagate` | `use_roi` | `dielectric_tensor_use_roi` | `multislice_propagation_roi` | `multislice_propagation_roi_merge_overlaps` | What it means |
|---|---:|---:|---:|---:|---:|---|
| Jones-only, full field | `False` | `False` | ignored | ignored | ignored | Local layer transmission everywhere. Slowest Jones path, useful as a reference. |
| Jones-only, aperture ROI | `False` | `True` | `True` | ignored | ignored | Local Jones interaction uses aperture support regions. Fast and usually the default for thin stacks. |
| Full multislice, full field | `True` | `False` | ignored | `False` | ignored | Full-field free-space FFT between slices. Most physically conservative, most expensive. |
| Full multislice with Jones/tensor ROI only | `True` | `True` | `True` | `False` | ignored | Jones/tensor work is ROI optimized, but free-space propagation is still full-field. Good accuracy/speed compromise. |
| ROI multislice, default merged crops | `True` | `True` | `True` | `True` | `True` | Free-space propagation uses padded aperture crops. Overlapping crops are merged before propagation, which is safer for close OH/RH layouts. |
| ROI multislice, separate-crop opt-out | `True` | `True` | `True` | `True` | `False` | Faster opt-out mode. Separate padded crops add their corrections independently, so use it only when padded boxes do not overlap or when the speed tradeoff is worth checking against a reference. |
| ROI multislice requested but no aperture ROIs | `True` | `False` | ignored | `True` | ignored | Falls back to full-field free-space propagation because no aperture support regions exist. |

The phrase “using ROIs for multislice but not for Jones” is mostly not a natural pipeline mode, because multislice ROI boxes come from the aperture support regions produced by the dielectric-tensor ROI path. In practice, if you disable aperture support generation, ROI multislice has nothing to crop around and falls back to full-field propagation.

Compatibility note: if this notebook is run with an older installed version of `scattering_calculator` that does not yet include `multislice_propagation_roi_merge_overlaps`, the code cells below will ignore that key and print a warning. In that case the separate-crop and merged-crop examples cannot be distinguished until the notebook kernel imports the updated local package.


In [ ]:
modes = {
    "jones_full_field": dict(
        propagate=False,
        use_roi=False,
        magnetic_pattern_use_roi=False,
        dielectric_tensor_use_roi=False,
        dielectric_tensor_compact=False,
        multislice_propagation_roi=False,
    ),
    "jones_aperture_roi": dict(
        propagate=False,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=False,
    ),
    "multislice_full_field": dict(
        propagate=True,
        use_roi=False,
        magnetic_pattern_use_roi=False,
        dielectric_tensor_use_roi=False,
        dielectric_tensor_compact=False,
        multislice_propagation_roi=False,
        propagation_padding_px=64,
        propagation_absorber_width_px=32,
        propagation_absorber_strength=4.0,
    ),
    "multislice_full_fft_with_jones_roi": dict(
        propagate=True,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=False,
        propagation_padding_px=64,
        propagation_absorber_width_px=32,
        propagation_absorber_strength=4.0,
    ),
    "multislice_aperture_roi_separate_crops": dict(
        propagate=True,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=True,
        multislice_propagation_roi_padding_px=16,
        # Explicit opt-out from the default. Use this only when padded aperture boxes are well separated,
        # or when you want to measure the speed/accuracy tradeoff against the merged-crop default.
        multislice_propagation_roi_merge_overlaps=False,
        propagation_padding_px=32,
        propagation_absorber_width_px=16,
        propagation_absorber_strength=4.0,
    ),
    "multislice_aperture_roi_merge_overlaps": dict(
        propagate=True,
        use_roi=True,
        magnetic_pattern_use_roi=True,
        dielectric_tensor_use_roi=True,
        dielectric_tensor_compact=True,
        multislice_propagation_roi=True,
        multislice_propagation_roi_padding_px=16,
        multislice_propagation_roi_merge_overlaps=True,
        propagation_padding_px=32,
        propagation_absorber_width_px=16,
        propagation_absorber_strength=4.0,
    ),
}

for name, settings in modes.items():
    print(name)
    for key, value in settings.items():
        print(f"  {key}: {value}")

CONFIG_FIELD_NAMES = {field.name for field in fields(HologramPipelineConfig)}
_warned_unsupported_mode_fields = set()


def compatible_mode_settings(settings, mode_name=None):
    """Drop mode keys unsupported by the HologramPipelineConfig imported by this kernel."""
    unsupported = {key: value for key, value in settings.items() if key not in CONFIG_FIELD_NAMES}
    if unsupported:
        warning_key = (mode_name, tuple(sorted(unsupported)))
        if warning_key not in _warned_unsupported_mode_fields:
            _warned_unsupported_mode_fields.add(warning_key)
            names = ", ".join(sorted(unsupported))
            label = f" for {mode_name}" if mode_name else ""
            print(
                f"Warning: the active HologramPipelineConfig does not support {names}{label}. "
                "Those settings will be ignored. Restart the notebook kernel or reinstall the local package "
                "if you expected the latest ROI-overlap behavior."
            )
    return {key: value for key, value in settings.items() if key in CONFIG_FIELD_NAMES}


def make_mode_config(mode_name):
    """Create a config for one tutorial mode, compatible with the active package version."""
    return replace(base_config, **compatible_mode_settings(modes[mode_name], mode_name=mode_name))


def config_value(config, name, default):
    """Read a possibly new config option without breaking older installed packages."""
    return getattr(config, name, default)


## 5. Padding and absorbers for multislice

When `propagate=True`, FFT propagation can suffer from wraparound artifacts at array boundaries. The pipeline exposes:

- `propagation_padding_px`: pad the field before each free-space FFT and crop back afterward.
- `propagation_padding_mode`: passed to NumPy padding; common choices are `"edge"` and `"reflect"`.
- `propagation_absorber_width_px`: apply a smooth absorber at padded edges.
- `propagation_absorber_strength`: stronger values damp edges more.
- `propagation_absorber_profile`: `"cosine"`, `"smoothstep"`, `"quadratic"`, or `"linear"`.

For ROI multislice, there are two different padding ideas:

- `multislice_propagation_roi_padding_px` grows the aperture ROI crop itself.
- `propagation_padding_px` pads inside each crop during FFT propagation.

On a small tutorial grid, a large `multislice_propagation_roi_padding_px` can expand the object-hole propagation crop until it covers nearly the full exit-wave frame. That is still a valid ROI run, but it will look almost like full-field multislice in the exit-wave plots. The example below uses a deliberately smaller ROI padding so the padded propagation boxes remain visible.


## 6. Tiny config for optional comparisons

The next cells define a small pipeline config. Running several modes can still take time, so the actual comparison is disabled by default. The example uses a binary labyrinth magnetic pattern so the CR/CL difference and sum channels contain visible spatial structure.


In [ ]:
output_folder = DATA_ROOT / "Data" / "multislice_roi_tutorial"
output_folder.mkdir(parents=True, exist_ok=True)

base_config = HologramPipelineConfig(
    recipe="Au(80)/Cr(5)/SiN(80)/Pt(4)Co(6)/Pt(2)",
    sample_name="multislice_roi_tutorial",
    xray_energy=778.0,
    xray_photon_flux=5e8,
    xray_coherence_length=(25e-6, 25e-6),
    detector_shape=(96, 96),
    detector_pixel_size=20e-6,
    detector_distance=0.03,
    detector_center=(48, 48),
    detector_params={
        "readout_noise_average": 20,
        "noise_rms": 3,
        "detector_threshold": 5e4,
        "counts_per_photon": 100,
        "quantum_efficiency": 0.9,
    },
    measurement_config={
        "number_frames": 1,
        "max_counts_per_image": 5e4,
        "exposure_time": 1.0,
    },
    beamstop_method="circular",
    beamstop_distance=0.010,
    beamstop_config={
        "radius": 180e-6,
        "sigma": 10e-6,
        "wire_width": 40e-6,
        "wire_bend": 30e-6,
        "angle": np.deg2rad(25),
        "antialias": 2,
        "seed": 4,
    },
    save_detected_hologram_without_beamstop=True,
    aperture_method="FTH_circular",
    aperture_types=["OH", "RH", "RH", "RH"],
    aperture_radii=[255e-9, 60e-9, 34e-9, 30e-9],
    aperture_centers=[(0.0, 0.0), (-660e-9, +60e-9), (-500e-9, -60e-9), (45e-9, -640e-9)],
    aperture_sigmas=[4e-9, 2e-9, 2e-9,2e-9],
    aperture_angles=[0.0, 0.0, 0.0,0],
    aperture_ellipticities=[1.0, 1.0, 1.0,1.0],
    aperture_roughnesses=[0.0, 0.02, 0.02,0.02],
    aperture_roughness_modes=[(0, 0), (3, 10), (3, 10), (3, 10)],
    aperture_seeds=[1, 2, 3,4],
    aperture_top_radius_factors=[1.3, 1.5, 1.75, 1.75],
    illumination_function="gaussian",
    illumination_center=(0.0, 0.0),
    illumination_focus_distance=1e-3,
    illumination_fwhm=0.45e-6,
    pattern_type="binary_labyrinth_pattern",
    pattern_config={
        "stripe_width": 75e-9,
        "sigma": 4e-9,
        "domain_conversion": "soft",
        "softness": 1.2,
        "n_steps": 60,
        "seed": 4,
        "use_gpu": False,
    },
    oversampling=2,
    random_seed=0,
)


## 7. Optional: run selected modes

Set `RUN_COMPARISON = True` to generate one small HDF5 file per selected mode. For a first run, compare only two or three modes.


In [ ]:
RUN_COMPARISON = True
selected_modes = [
    "jones_aperture_roi",
    "multislice_full_fft_with_jones_roi",
    "multislice_aperture_roi_separate_crops",
    "multislice_aperture_roi_merge_overlaps",
]

results = {}
if RUN_COMPARISON:
    for mode_name in selected_modes:
        cfg = make_mode_config(mode_name)
        output_path = output_folder / f"{mode_name}.h5"
        t0 = time.time()
        HologramPipeline(
            config=cfg,
            ranges=HologramPipelineRanges(),
            output_path=output_path,
            n_samples=1,
            verbose=False,
        ).run()
        elapsed = time.time() - t0
        results[mode_name] = {"path": output_path, "seconds": elapsed}
        print(f"{mode_name}: {elapsed:.2f} s -> {output_path}")
else:
    print("Comparison run is disabled. Set RUN_COMPARISON = True to generate mode outputs.")


## 8. Load and compare outputs

The comparison below displays the modes you selected in section 7. It shows both helicity **differences** (`CR - CL`) and helicity **sums** (`CR + CL`). Differences emphasize magnetic contrast; sums emphasize charge/background structure. For complex exit waves, both logarithmic intensity and phase are shown.

Rectangle overlays have two meanings:

- Solid cyan rectangles estimate the unpadded aperture support boxes from the saved support mask. These are shown for all modes that saved a support mask.
- Dashed lime rectangles are shown only when `multislice_propagation_roi=True`. They estimate the padded free-space propagation boxes used by ROI multislice. If `multislice_propagation_roi_merge_overlaps=True`, overlapping padded boxes are merged before both plotting and local propagation. If it is `False`, separate padded crops are shown and their corrections are accumulated independently.

These rectangles are computational regions, not hard exit-wave masks. Full-field modes can show diffraction everywhere, and ROI modes can still carry the plane-wave/background field outside the dashed boxes. The printed ROI flags and coverage estimates below confirm which mode each HDF5 file actually used.


In [ ]:
def first_frame(array):
    array = np.asarray(array)
    return array[0] if array.ndim == 3 else array


def read_optional_scalar(group, key, default=None):
    if key not in group:
        return default
    value = group[key][()]
    if isinstance(value, np.generic):
        return value.item()
    return value


def load_mode_output(path):
    path = Path(path)
    if not path.exists():
        return None
    with h5py.File(path, "r") as h5:
        grp = h5["00000"]
        roi_flags = {
            "pipeline_propagate": bool(read_optional_scalar(h5["_pipeline_config"], "propagate", False)),
            "pipeline_multislice_roi": bool(read_optional_scalar(h5["_pipeline_config"], "multislice_propagation_roi", False)),
            "pipeline_multislice_roi_padding_px": int(read_optional_scalar(h5["_pipeline_config"], "multislice_propagation_roi_padding_px", 0)),
            "pipeline_multislice_roi_merge_overlaps": bool(read_optional_scalar(h5["_pipeline_config"], "multislice_propagation_roi_merge_overlaps", True)),
            "sample_use_roi": bool(read_optional_scalar(grp["metadata"], "use_roi", False)),
            "magnetic_pattern_roi": bool(read_optional_scalar(grp["metadata"], "magnetic_pattern/use_roi", False)),
            "dielectric_tensor_roi": bool(read_optional_scalar(grp["metadata"], "dielectric_tensor/use_roi", False)),
            "metadata_multislice_roi": bool(read_optional_scalar(grp["metadata"], "propagation/multislice_roi", False)),
            "metadata_multislice_roi_merge_overlaps": bool(read_optional_scalar(grp["metadata"], "propagation/multislice_roi_merge_overlaps", True)),
        }
        return {
            "CR_detected": first_frame(grp["CR/detected"][()]),
            "CL_detected": first_frame(grp["CL/detected"][()]),
            "CR_exit": first_frame(grp["CR/exit_wave"][()]),
            "CL_exit": first_frame(grp["CL/exit_wave"][()]),
            "beamstop": grp["beamstop_mask"][()] if "beamstop_mask" in grp else None,
            "supportmask": grp["supportmask"][()] if "supportmask" in grp else None,
            "roi_flags": roi_flags,
        }


def robust_limits(image, percentiles=(1, 99)):
    data = np.asarray(image)
    vmin, vmax = np.nanpercentile(data, percentiles)
    if np.isclose(vmin, vmax):
        vmin, vmax = float(np.nanmin(data)), float(np.nanmax(data))
    if np.isclose(vmin, vmax):
        vmin, vmax = vmin - 0.5, vmax + 0.5
    return vmin, vmax


def log_intensity(image, floor_percentile=0.1):
    """Return log10 intensity/counts with a robust positive floor."""
    data = np.asarray(image, dtype=float)
    positive = data[data > 0]
    if positive.size == 0:
        floor = 1.0
    else:
        floor = np.percentile(positive, floor_percentile)
        if floor <= 0:
            floor = np.min(positive)
    return np.log10(np.clip(data, floor, None))


def supportmask_roi_boxes(supportmask):
    """Estimate ROI boxes from connected regions in the saved support mask."""
    if supportmask is None:
        return []
    mask = np.asarray(supportmask) > 0
    if not np.any(mask):
        return []

    try:
        from scipy.ndimage import label, find_objects

        labels, _ = label(mask)
        objects = find_objects(labels)
        return [obj for obj in objects if obj is not None]
    except Exception:
        y, x = np.where(mask)
        return [(slice(int(y.min()), int(y.max()) + 1), slice(int(x.min()), int(x.max()) + 1))]


def scale_roi_boxes(boxes, source_shape, target_shape):
    """Scale aperture ROI boxes from one grid to another."""
    scale_y = target_shape[0] / source_shape[0]
    scale_x = target_shape[1] / source_shape[1]
    scaled = []
    for y_slice, x_slice in boxes:
        scaled.append(
            (
                slice(int(round(y_slice.start * scale_y)), int(round(y_slice.stop * scale_y))),
                slice(int(round(x_slice.start * scale_x)), int(round(x_slice.stop * scale_x))),
            )
        )
    return scaled


def pad_roi_boxes(boxes, target_shape, padding_px):
    """Pad ROI boxes in target-grid pixels and clip to the target shape."""
    pad = max(0, int(padding_px))
    padded = []
    for y_slice, x_slice in boxes:
        padded.append(
            (
                slice(max(0, y_slice.start - pad), min(target_shape[0], y_slice.stop + pad)),
                slice(max(0, x_slice.start - pad), min(target_shape[1], x_slice.stop + pad)),
            )
        )
    return padded


def merge_overlapping_roi_boxes(boxes):
    """Merge overlapping ROI boxes to match the propagator crop logic."""
    merged = []
    for y_slice, x_slice in boxes:
        pending = [y_slice.start, y_slice.stop, x_slice.start, x_slice.stop]
        i = 0
        while i < len(merged):
            current = merged[i]
            overlaps = (
                pending[0] < current[1]
                and current[0] < pending[1]
                and pending[2] < current[3]
                and current[2] < pending[3]
            )
            if overlaps:
                merged.pop(i)
                pending = [
                    min(pending[0], current[0]),
                    max(pending[1], current[1]),
                    min(pending[2], current[2]),
                    max(pending[3], current[3]),
                ]
                i = 0
            else:
                i += 1
        merged.append(pending)
    merged.sort(key=lambda item: (item[0], item[2], item[1], item[3]))
    return [(slice(y0, y1), slice(x0, x1)) for y0, y1, x0, x1 in merged]


def roi_coverage_fraction(boxes, target_shape):
    """Return the fraction of target pixels covered by the union of ROI boxes."""
    if not boxes:
        return 0.0
    mask = np.zeros(target_shape[:2], dtype=bool)
    for y_slice, x_slice in boxes:
        mask[y_slice, x_slice] = True
    return float(mask.mean())


def draw_boxes(ax, boxes, color="cyan", linestyle="-", linewidth=1.5):
    """Overlay ROI boxes already expressed in the plotted image grid."""
    for y_slice, x_slice in boxes:
        ax.add_patch(
            plt.Rectangle(
                (x_slice.start - 0.5, y_slice.start - 0.5),
                x_slice.stop - x_slice.start,
                y_slice.stop - y_slice.start,
                fill=False,
                edgecolor=color,
                linestyle=linestyle,
                linewidth=linewidth,
            )
        )


def draw_roi_boxes(ax, boxes, source_shape, target_shape=None, color="cyan"):
    """Overlay aperture ROI boxes, scaling from source grid to target grid."""
    if target_shape is None:
        target_shape = source_shape
    draw_boxes(ax, scale_roi_boxes(boxes, source_shape, target_shape), color=color)


loaded = {
    mode_name: load_mode_output(output_folder / f"{mode_name}.h5")
    for mode_name in selected_modes
}
loaded = {name: data for name, data in loaded.items() if data is not None}

if not loaded:
    print("No comparison outputs found yet. Set RUN_COMPARISON = True in the previous section.")
else:
    fig, axes = plt.subplots(len(loaded), 7, figsize=(22, 3.5 * len(loaded)))
    if len(loaded) == 1:
        axes = axes[np.newaxis, :]
    for mode_name, data in loaded.items():
        supportmask = data["supportmask"]
        exit_shape = data["CR_exit"].shape
        support_shape = np.asarray(supportmask).shape if supportmask is not None else exit_shape
        support_boxes = scale_roi_boxes(supportmask_roi_boxes(supportmask), support_shape, exit_shape)
        padding_px = data["roi_flags"].get("pipeline_multislice_roi_padding_px", 0)
        propagation_boxes = pad_roi_boxes(support_boxes, exit_shape, padding_px)
        if data["roi_flags"].get("pipeline_multislice_roi_merge_overlaps", False):
            propagation_boxes = merge_overlapping_roi_boxes(propagation_boxes)
        coverage = roi_coverage_fraction(propagation_boxes, exit_shape)
        merge_label = "padded+merged" if data["roi_flags"].get("pipeline_multislice_roi_merge_overlaps", False) else "padded separate"
        print(mode_name, data["roi_flags"], f"estimated {merge_label} propagation ROI coverage: {coverage:.1%}")

    for row, (mode_name, data) in enumerate(loaded.items()):
        detected_diff = data["CR_detected"] - data["CL_detected"]
        detected_sum_log = log_intensity(data["CR_detected"] + data["CL_detected"])
        exit_diff = data["CR_exit"] - data["CL_exit"]
        exit_sum = data["CR_exit"] + data["CL_exit"]
        exit_diff_log_intensity = log_intensity(np.abs(exit_diff) ** 2)
        exit_sum_log_intensity = log_intensity(np.abs(exit_sum) ** 2)
        supportmask = data["supportmask"]
        roi_boxes = supportmask_roi_boxes(supportmask)
        support_shape = np.asarray(supportmask).shape if supportmask is not None else exit_diff.shape
        padding_px = data["roi_flags"].get("pipeline_multislice_roi_padding_px", 0)
        panels = [
            (detected_diff, "detected CR - CL", "RdBu_r", None, False),
            (detected_sum_log, "log10 detected CR + CL", "magma", None, False),
            (exit_diff_log_intensity, "log10 intensity exit CR - CL", "inferno", None, True),
            (np.angle(exit_diff), "phase exit wave CR - CL", "hsv", (-np.pi, np.pi), True),
            (exit_sum_log_intensity, "log10 intensity exit CR + CL", "viridis", None, True),
            (np.angle(exit_sum), "phase exit wave CR + CL", "hsv", (-np.pi, np.pi), True),
            (supportmask, "support mask", "gray", (0, 1), True),
        ]
        for ax, (image, title, cmap, fixed_limits, mark_rois) in zip(axes[row], panels):
            if fixed_limits is None:
                vmin, vmax = robust_limits(image)
            else:
                vmin, vmax = fixed_limits
            ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
            if mark_rois:
                target_shape = np.asarray(image).shape
                support_boxes = scale_roi_boxes(roi_boxes, support_shape, target_shape)
                draw_boxes(ax, support_boxes, color="cyan", linestyle="-", linewidth=1.4)
                if data["roi_flags"].get("pipeline_multislice_roi", False):
                    padded_boxes = pad_roi_boxes(support_boxes, target_shape, padding_px)
                    if data["roi_flags"].get("pipeline_multislice_roi_merge_overlaps", False):
                        padded_boxes = merge_overlapping_roi_boxes(padded_boxes)
                    draw_boxes(ax, padded_boxes, color="lime", linestyle="--", linewidth=1.2)
            ax.set_title(f"{mode_name}\n{title}")
            ax.set_axis_off()


## 9. Vertical x-z propagation slices by mode

The 2-D exit-wave plots above show the field after the sample. For intuition, it is also useful to look at vertical sections through the apertures and compare the propagation modes directly.

The diagnostic below rebuilds the same aperture mask and layer stack used by each mode configuration. For every requested aperture, it plots a row per mode: Jones-only modes keep the line local from layer to layer, full multislice propagates the whole line between layers, and ROI multislice propagates only padded aperture intervals while the rest of the line receives the plane-wave baseline phase.

This is a teaching visualization, not a new saved pipeline observable. The production solver propagates a 2-D Jones wavefield; this cell compresses the picture to one transverse line so that the z evolution and ROI approximation are easy to inspect.


In [ ]:
def rebuild_aperture_for_vertical_diagnostic(config):
    """Rebuild the tutorial aperture mask on the same grid used by HologramPipeline."""
    wavelength = physics.photon_energy_wavelength(config.xray_energy, unit="eV")
    real_space_resolution = (
        wavelength * config.detector_distance / (config.detector_pixel_size * config.detector_shape[1])
    )
    real_space_pixel_size = real_space_resolution / config.oversampling
    sample_shape = np.array(
        [0, config.oversampling * config.detector_shape[0], config.oversampling * config.detector_shape[1]],
        dtype=int,
    )

    xray_config = XRayConfig(
        energy=config.xray_energy,
        pol="CR",
        photon_flux=config.xray_photon_flux,
        coherence_length=config.xray_coherence_length,
    )
    xray_config.setup()

    sample_config = SampleConfig(
        recipe=config.recipe,
        sample_shape=sample_shape,
        real_space_pixel_size=real_space_pixel_size,
        xray_config=xray_config,
        sample_name=config.sample_name,
    )
    sample_config.setup()

    layer_names = list(sample_config.sample_structure.layer_names)
    layer_thicknesses = np.asarray(sample_config.sample_structure.layer_thicknesses, dtype=float)
    membrane_index = layer_names.index("SiN")
    aperture_taper_depth = float(np.sum(layer_thicknesses[: max(0, membrane_index - 2)]))
    thickness_oh = float(np.sum(layer_thicknesses[:membrane_index]))

    aperture_config = dict(
        apertures_type=config.aperture_types,
        apertures_radius=config.aperture_radii,
        apertures_center=config.aperture_centers,
        apertures_sigma=config.aperture_sigmas,
        apertures_angle=config.aperture_angles,
        apertures_ellipticity=config.aperture_ellipticities,
        apertures_roughness=config.aperture_roughnesses,
        apertures_roughness_modes=config.aperture_roughness_modes,
        apertures_seed=config.aperture_seeds,
        apertures_top_radius_factor=config.aperture_top_radius_factors,
        aperture_taper_depth=aperture_taper_depth,
        thickness_OH=thickness_oh,
    )
    front_aperture = FrontApertureConfig(
        aperture_method=config.aperture_method,
        aperture_shape=sample_config.sample_structure.sample_shape,
        real_space_pixel_size=real_space_pixel_size,
        aperture_thicknesses=layer_thicknesses,
        aperture_config=aperture_config,
        use_roi=config.use_roi,
    )
    front_aperture.setup()
    return {
        "aperture_mask": front_aperture.return_aperture(),
        "layer_names": layer_names,
        "layer_thicknesses": layer_thicknesses,
        "real_space_pixel_size": real_space_pixel_size,
        "wavelength": wavelength,
    }


def propagate_line_angular_spectrum(field, wavelength, dz, pixel_size):
    """One-dimensional angular-spectrum propagation used only for this x-z diagnostic."""
    if dz == 0:
        return field.copy()
    k0 = 2 * np.pi / wavelength
    fx = np.fft.fftfreq(field.size, d=pixel_size)
    kx2 = (2 * np.pi * fx) ** 2
    propagating = kx2 <= k0**2
    kz = np.zeros_like(kx2, dtype=float)
    kz[propagating] = np.sqrt(np.maximum(k0**2 - kx2[propagating], 0.0))
    decay = np.ones_like(kx2, dtype=float)
    if np.any(~propagating):
        decay[~propagating] = np.exp(-np.sqrt(kx2[~propagating] - k0**2) * abs(dz))
    kernel = np.exp(-1j * kz * dz) * decay
    return np.fft.ifft(np.fft.fft(field) * kernel)


def line_intervals_from_open_fraction(open_fraction, padding_px=0):
    """Return padded 1-D intervals where the aperture is open along this x cut."""
    mask = np.asarray(open_fraction) > 0.02
    if not np.any(mask):
        return []
    edges = np.diff(np.r_[False, mask, False].astype(int))
    starts = np.where(edges == 1)[0]
    stops = np.where(edges == -1)[0]
    pad = max(0, int(padding_px))
    n = mask.size
    return [(max(0, int(s) - pad), min(n, int(e) + pad)) for s, e in zip(starts, stops)]


def merge_line_intervals(intervals):
    """Merge overlapping 1-D propagation intervals, mirroring merged ROI boxes."""
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for start, stop in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], stop)
        else:
            merged.append([start, stop])
    return [tuple(item) for item in merged]


def propagate_line_by_mode(field, open_fraction, config, wavelength, dz, pixel_size):
    """Apply the inter-layer propagation rule for one tutorial mode."""
    if not config.propagate:
        return field

    if not config.multislice_propagation_roi:
        return propagate_line_angular_spectrum(field, wavelength, dz, pixel_size)

    baseline = field * np.exp(-1j * (2 * np.pi / wavelength) * dz)
    intervals = line_intervals_from_open_fraction(
        open_fraction,
        padding_px=config_value(config, "multislice_propagation_roi_padding_px", 0),
    )
    if config_value(config, "multislice_propagation_roi_merge_overlaps", True):
        intervals = merge_line_intervals(intervals)
    for start, stop in intervals:
        local_out = propagate_line_angular_spectrum(field[start:stop], wavelength, dz, pixel_size)
        baseline[start:stop] += local_out - field[start:stop] * np.exp(-1j * (2 * np.pi / wavelength) * dz)
    return baseline


def simulate_vertical_wave_slice(geometry, config, aperture_index=0):
    """Propagate a didactic 1-D field through an x-z aperture cut for one mode."""
    aperture_mask = geometry["aperture_mask"]
    pixel_size = geometry["real_space_pixel_size"]
    wavelength = geometry["wavelength"]
    layer_thicknesses = geometry["layer_thicknesses"]
    ny, nx = aperture_mask.shape[1:]
    center_y_m, center_x_m = config.aperture_centers[aperture_index]
    cut_y = int(np.clip(round(ny / 2 + center_y_m / pixel_size), 0, ny - 1))
    x_m = (np.arange(nx) - nx / 2) * pixel_size

    sigma = config.illumination_fwhm / (2 * np.sqrt(2 * np.log(2)))
    y_rel = center_y_m - config.illumination_center[0]
    x_rel = x_m - config.illumination_center[1]
    field = np.exp(-(x_rel**2 + y_rel**2) / (2 * sigma**2)).astype(complex)
    if config.illumination_focus_distance:
        field *= np.exp(-1j * np.pi * (x_rel**2 + y_rel**2) / (wavelength * config.illumination_focus_distance))

    rows = []
    intervals_by_layer = []
    for iz, dz in enumerate(layer_thicknesses):
        material_fraction = aperture_mask[iz, cut_y, :]
        open_fraction = 1.0 - material_fraction
        # This is a visual aperture gate, not a material-parameter calculation.
        # It keeps a tiny background amplitude so the phase map remains readable outside the hole.
        field = field * (0.03 + 0.97 * open_fraction)
        rows.append(field.copy())

        intervals = []
        if config.propagate and config.multislice_propagation_roi:
            intervals = line_intervals_from_open_fraction(
                open_fraction,
                padding_px=config_value(config, "multislice_propagation_roi_padding_px", 0),
            )
            if config_value(config, "multislice_propagation_roi_merge_overlaps", True):
                intervals = merge_line_intervals(intervals)
        intervals_by_layer.append(intervals)

        if iz < len(layer_thicknesses) - 1:
            field = propagate_line_by_mode(field, open_fraction, config, wavelength, dz, pixel_size)

    return {
        "x_nm": x_m * 1e9,
        "z_edges_nm": np.concatenate([[0.0], np.cumsum(layer_thicknesses) * 1e9]),
        "z_centers_nm": (np.cumsum(layer_thicknesses) - 0.5 * layer_thicknesses) * 1e9,
        "field_xz": np.asarray(rows),
        "open_fraction_xz": 1.0 - aperture_mask[:, cut_y, :],
        "intervals_by_layer": intervals_by_layer,
        "cut_y": cut_y,
    }


def draw_material_borders(ax, layer_names, z_edges_nm):
    for z in z_edges_nm:
        ax.axhline(z, color="white", lw=0.6, alpha=0.45)
    for iz, name in enumerate(layer_names):
        z0, z1 = z_edges_nm[iz], z_edges_nm[iz + 1]
        if name == "SiN":
            color = "yellow"
            label = "SiN borders"
        elif "Co" in name or "xmcd" in name.lower():
            color = "magenta"
            label = "magnetic-material borders"
        else:
            continue
        ax.axhline(z0, color=color, lw=1.6, alpha=0.95, label=label)
        ax.axhline(z1, color=color, lw=1.6, alpha=0.95)


def draw_vertical_roi_intervals(ax, intervals_by_layer, z_edges_nm, x_nm, pixel_size_nm):
    """Mark the 1-D intervals used by ROI multislice propagation."""
    for iz, intervals in enumerate(intervals_by_layer):
        z0, z1 = z_edges_nm[iz], z_edges_nm[iz + 1]
        for start, stop in intervals:
            left = x_nm[start] - 0.5 * pixel_size_nm
            right = x_nm[stop - 1] + 0.5 * pixel_size_nm
            ax.add_patch(
                plt.Rectangle(
                    (left, z0),
                    right - left,
                    z1 - z0,
                    fill=False,
                    edgecolor="lime",
                    linestyle="--",
                    linewidth=0.9,
                    alpha=0.85,
                )
            )


def mode_label(config):
    if not config.propagate:
        return "Jones only"
    if not config.multislice_propagation_roi:
        return "full-line multislice"
    if config_value(config, "multislice_propagation_roi_merge_overlaps", True):
        return "ROI multislice, merged intervals"
    return "ROI multislice, separate intervals"


def plot_vertical_wave_slices_by_mode(mode_configs, aperture_indices=(0, 1, 2, 3)):
    """Plot amplitude and phase x-z sections for each aperture and propagation mode."""
    geometry = rebuild_aperture_for_vertical_diagnostic(next(iter(mode_configs.values())))
    layer_names = geometry["layer_names"]
    z_edges_nm = np.concatenate([[0.0], np.cumsum(geometry["layer_thicknesses"]) * 1e9])
    pixel_size_nm = geometry["real_space_pixel_size"] * 1e9

    for aperture_index in aperture_indices:
        if aperture_index >= len(base_config.aperture_centers):
            continue
        n_modes = len(mode_configs)
        fig, axes = plt.subplots(n_modes, 2, figsize=(13, 2.8 * n_modes), sharex=True, sharey=True)
        axes = np.atleast_2d(axes)
        aperture_label = f"{base_config.aperture_types[aperture_index]} {aperture_index}"
        for row, (mode_name, config) in enumerate(mode_configs.items()):
            section = simulate_vertical_wave_slice(geometry, config, aperture_index=aperture_index)
            x_nm = section["x_nm"]
            z_centers_nm = section["z_centers_nm"]
            field = section["field_xz"]
            open_fraction = section["open_fraction_xz"]
            x_edges_nm = np.r_[x_nm - 0.5 * pixel_size_nm, x_nm[-1] + 0.5 * pixel_size_nm]
            amplitude = np.abs(field)
            amplitude = amplitude / max(np.nanmax(amplitude), 1e-12)
            phase = np.angle(field)

            amp_ax, phase_ax = axes[row]
            im0 = amp_ax.pcolormesh(x_edges_nm, z_edges_nm, amplitude, shading="auto", cmap="magma", vmin=0, vmax=1)
            im1 = phase_ax.pcolormesh(x_edges_nm, z_edges_nm, phase, shading="auto", cmap="twilight", vmin=-np.pi, vmax=np.pi)
            for ax in (amp_ax, phase_ax):
                ax.contour(x_nm, z_centers_nm, open_fraction, levels=[0.5], colors="cyan", linewidths=1.2)
                if config.propagate and config.multislice_propagation_roi:
                    draw_vertical_roi_intervals(ax, section["intervals_by_layer"], z_edges_nm, x_nm, pixel_size_nm)
                draw_material_borders(ax, layer_names, z_edges_nm)
                ax.set_ylim(z_edges_nm[-1], 0)
                ax.set_ylabel("depth z (nm)")
            amp_ax.set_title(f"{mode_name}\n{mode_label(config)}: amplitude")
            phase_ax.set_title(f"{mode_name}\n{mode_label(config)}: phase")
            fig.colorbar(im0, ax=amp_ax, fraction=0.046, pad=0.02, label="|E| / max")
            fig.colorbar(im1, ax=phase_ax, fraction=0.046, pad=0.02, label="phase (rad)")

        for ax in axes[-1]:
            ax.set_xlabel("x (nm)")
        handles, labels = axes[0, 0].get_legend_handles_labels()
        if handles:
            by_label = dict(zip(labels, handles))
            fig.legend(by_label.values(), by_label.keys(), loc="upper right")
        fig.suptitle(
            f"Didactic x-z wavefront sections through {aperture_label}: aperture wall cyan, ROI intervals dashed lime"
        )


vertical_mode_names = [
    "jones_full_field",
    "jones_aperture_roi",
    "multislice_full_field",
    "multislice_full_fft_with_jones_roi",
    "multislice_aperture_roi_separate_crops",
    "multislice_aperture_roi_merge_overlaps",
]
vertical_mode_configs = {
    name: make_mode_config(name)
    for name in vertical_mode_names
}
plot_vertical_wave_slices_by_mode(vertical_mode_configs, aperture_indices=range(len(base_config.aperture_centers)))


## 10. Recommended choices

For most users:

- Start with `propagate=False`, `use_roi=True`, `dielectric_tensor_use_roi=True`, `dielectric_tensor_compact=True`.
- Turn on `propagate=True` when the sample stack is thick enough that propagation between layers matters.
- With `propagate=True`, first try `multislice_propagation_roi=False` so free-space propagation remains full-field.
- Use `multislice_propagation_roi=True` for large sweeps where speed matters and the aperture holes occupy only a small part of the sample plane.
- Increase `multislice_propagation_roi_padding_px` if ROI-only multislice creates edge artifacts around apertures.
- `multislice_propagation_roi_merge_overlaps=True` is the default and is recommended when padded RH/OH boxes overlap. Set it to `False` only when you explicitly want the faster separate-crop approximation.
- Use `use_roi=False` for reference/debugging runs, small arrays, or when you want to compare against the most direct full-field calculation.

The safest comparison workflow is to run one configuration in two modes, compare CR-CL holograms and exit waves, and only then launch a large sweep.
